<a href="https://colab.research.google.com/github/Joaoplims/NLP_Gametox/blob/main/NLP_Gametox.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#TODO


## 📋 Checklist de Execução do Experimento

### 🛠️ 1. Preparação dos Dados Reais (GameTox)

* [ ] **Carga e Filtragem:** Carregar o dataset do GameTox e filtrar as 42.963 mensagens pertencentes ao subconjunto em inglês.


* [ ] **Inspeção das Classes:** Mapear a contagem exata das 6 classes de *Intent* (*Non-Toxic*, *Insults and Flaming*, *Other Offensive Texts*, *Hate and Harassment*, *Threats*, *Extremism*) para documentar a linha de base do desbalanceamento.


* [ ] **Pré-processamento e Normalização:**
* [ ] Converter todo o texto para caixa baixa (*lowercase*).


* [ ] Tratar/remover caracteres especiais, pontuações desnecessárias.


* [ ] Criar uma rotina de tokenização adequada preservando pontuações e expressões típicas do contexto *gamer* (ex: *n00b*, *kys*, *fck*).


* [ ] **Garantia de Integridade:** Isolar o conjunto de **Teste Real** (nenhum dado sintético deve entrar aqui sob hipótese alguma).





---

### ⚙️ 2. Construção da Baseline (Sem Dados Sintéticos)

* [ ] **Vetorização dos Dados de Treino:**
* [ ] Extrair representações **TF-IDF** (testando uni-grams e bi-grams).
* [ ] Extrair/gerar representações densas via **Word2Vec** (treinado no próprio corpus de treino ou usando um modelo pré-treinado adaptado).


* [ ] **Treinamento de Modelos Clássicos:**
* [ ] Treinar os modelos selecionados (ex: *Regressão Logística*, *SVM*, *Naive Bayes*, *Random Forest*) apenas com o **Treino Real Desbalanceado**.




* [ ] **Avaliação e Métricas de Referência:**
* [ ] Avaliar a baseline no conjunto de **Teste Real**.


* [ ] Registrar o **Macro F1-Score**, o **F1-Score individual por classe** e a **Matriz de Confusão** (espera-se desempenho muito baixo ou nulo nas classes raras: *Threats*, *Extremism*, *Hate*).





---

### 🤖 3. Geração e Validação de Dados Sintéticos (LLM)

* [ ] **Engenharia de Prompt:**
* [ ] Escrever prompts estruturados (*Single-Agent* ou *Multi-Agent*) fornecendo exemplos reais (*few-shot*) das classes minoritárias (*Threats*, *Extremism*, *Hate and Harassment*).


* [ ] Instruir a LLM a simular a linguagem informal, ruidosa e cheia de gírias do ambiente de jogos digitais.




* [ ] **Geração e Filtragem:**
* [ ] Gerar amostras sintéticas suficientes para equilibrar proporcionalmente as classes raras em relação às classes majoritárias no conjunto de treino.


* [ ] Fazer uma checagem de qualidade/limpeza no texto gerado (remover saídas polidas demais ou respostas que fujam do formato do chat).




* [ ] **Consolidação das Bases de Treino:**
* [ ] **Treino A (Baseline):** 100% Real.


* [ ] **Treino B (Tradicional):** Real + SMOTE / Random Oversampling nos vetores.
* [ ] **Treino C (Proposta LLM):** Real + Dados Sintéticos gerados.





---

### 🧪 4. Experimentos Comparativos e Treinamento

* [ ] **Re-vetorizar** os novos conjuntos de treino estendidos (B e C) utilizando exatamente os mesmos parâmetros de TF-IDF e Word2Vec definidos no Passo 2.
* [ ] **Treinar os classificadores** em cada um dos cenários de treino (A, B e C) mantendo os mesmos hiperparâmetros.

---

### 📊 5. Avaliação Final, Análise e Discussão

* [ ] **Avaliação Padronizada:** Avaliar todos os modelos resultantes exatamente sobre o mesmo conjunto de **Teste Real** (mantido intacto).


* [ ] **Métricas e Tabelas:**
* [ ] Gerar a tabela comparativa contendo: **Precision**, **Recall**, **Macro F1-Score** e **F1-Score por classe** para cada modelo/cenário.


* [ ] Plotar as **Matrizes de Confusão** comparativas (Baseline vs. SMOTE vs. Sintético LLM).




* [ ] **Análise de Erros (Qualitativa):**
* [ ] Analisar falsos positivos e falsos negativos: O dado sintético ajudou o modelo a generalizar sem causar overfit ao estilo do LLM?


* [ ] Verificar se gírias neutras de jogos foram confundidas com frases tóxicas sintéticas.


# Dependencias:

In [14]:
pip install gensim

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 27.9/27.9 MB 55.1 MB/s eta 0:00:00


# Datasets


In [3]:
val = "https://raw.githubusercontent.com/Joaoplims/NLP_Gametox/refs/heads/main/val-gametox/val/val.csv"
train = "https://raw.githubusercontent.com/Joaoplims/NLP_Gametox/refs/heads/main/train-gametox/train/train.csv"
test_label = "https://raw.githubusercontent.com/Joaoplims/NLP_Gametox/refs/heads/main/test_release-gametox/test_release/test_index_label.csv"
test_msg = "https://raw.githubusercontent.com/Joaoplims/NLP_Gametox/refs/heads/main/test_release-gametox/test_release/test_index_text.csv"

In [5]:
import pandas as pd
import re

# 1. Carregar os CSVs originais
df_train = pd.read_csv(train)
df_val = pd.read_csv(val)
df_test_msgs = pd.read_csv(test_msg)
df_test_lbls = pd.read_csv(test_label)
df_test = pd.merge(df_test_msgs, df_test_lbls, on='index')

# Função para checar se a mensagem NÃO contém caracteres cirílicos
def is_clean_english(text):
    return not bool(re.search(r'[\u0400-\u04FF]', str(text)))

# 2. Filtrar cada split
train_clean = df_train[df_train['message'].apply(is_clean_english)].copy()
val_clean   = df_val[df_val['message'].apply(is_clean_english)].copy()
test_clean  = df_test[df_test['message'].apply(is_clean_english)].copy()

# 3. Salvar os datasets filtrados
train_clean.to_csv('train_clean.csv', index=False)
val_clean.to_csv('val_clean.csv', index=False)
test_clean.to_csv('test_clean.csv', index=False)

print("=== SPLITS LIMPOS TEXTOS EM INGLÊS ===")
print(f"Treino:     {len(train_clean)} amostras")
print(f"Validação:  {len(val_clean)} amostras")
print(f"Teste Real: {len(test_clean)} amostras")
print(f"Total:      {len(train_clean) + len(val_clean) + len(test_clean)} amostras")

=== SPLITS LIMPOS TEXTOS EM INGLÊS ===
Treino:     39995 amostras
Validação:  4986 amostras
Teste Real: 4986 amostras
Total:      49967 amostras


In [6]:
import pandas as pd

# Mapeamento das classes conforme a documentação do GameTox
class_mapping = {
    0.0: "Non-Toxic",
    1.0: "Insults and Flaming",
    2.0: "Other Offensive Texts",
    3.0: "Hate and Harassment",
    4.0: "Threats",
    5.0: "Extremism"
}

# Contagem no conjunto de treino limpo
class_counts = train_clean['label'].value_counts().sort_index().rename(index=class_mapping)
class_percentages = (train_clean['label'].value_counts(normalize=True) * 100).sort_index().rename(index=class_mapping)

# Criando um DataFrame para visualização
imbalance_report = pd.DataFrame({
    'Amostras': class_counts,
    'Porcentagem (%)': class_percentages.map('{:.2f}%'.format)
})

print("=== DISTRIBUIÇÃO DAS CLASSES (TRAIN_CLEAN) ===")
print(imbalance_report)

=== DISTRIBUIÇÃO DAS CLASSES (TRAIN_CLEAN) ===
                       Amostras Porcentagem (%)
label                                          
Non-Toxic                 32511          81.29%
Insults and Flaming        5435          13.59%
Other Offensive Texts      1755           4.39%
Hate and Harassment         216           0.54%
Threats                      55           0.14%
Extremism                    23           0.06%


# Pre processamento do Dataset

In [7]:
import re

def preprocess_gamer_text(text):
    if not isinstance(text, str):
        return ""
    # Converter para caixa baixa
    text = text.lower()
    # Remover caracteres especiais, mas manter o que é comum em chats (letras, números e espaços)
    # Mantemos alguns símbolos que podem ser usados em gírias ou censura (como *)
    text = re.sub(r'[^a-z0-9\s\*]', '', text)
    # Remover espaços extras
    text = re.sub(r'\s+', ' ', text).strip()
    return text

# Aplicando a limpeza nos DataFrames limpos
train_clean['message_preprocessed'] = train_clean['message'].apply(preprocess_gamer_text)
val_clean['message_preprocessed'] = val_clean['message'].apply(preprocess_gamer_text)
test_clean['message_preprocessed'] = test_clean['message'].apply(preprocess_gamer_text)

print("=== PRÉ-PROCESSAMENTO CONCLUÍDO ===")
print("Exemplos de antes e depois (Treino):")
display(train_clean[['message', 'message_preprocessed']].head(10))

=== PRÉ-PROCESSAMENTO CONCLUÍDO ===
Exemplos de antes e depois (Treino):


,message,message_preprocessed
0,no rush,no rush
1,whatever ... watch the replay,whatever watch the replay
2,useless,useless
3,3 gunmark,3 gunmark
4,lol,lol
5,i softened him lol,i softened him lol
6,come,come
7,stupid kids grow up,stupid kids grow up
8,fking light diference,fking light diference
9,"hori, info ?",hori info


# Vetorização

## TF-IDF

In [9]:
from sklearn.feature_extraction.text import TfidfVectorizer

# Inicializar o TF-IDF Vectorizer
# Usaremos unigramas e bigramas (ngram_range=(1, 2)) e limitaremos a 10.000 features para eficiência
tfidf_vect = TfidfVectorizer(ngram_range=(1, 2), max_features=10000)

# 1. Ajustar e transformar o conjunto de TREINO
X_train_tfidf = tfidf_vect.fit_transform(train_clean['message_preprocessed'])

# 2. Apenas transformar os conjuntos de VALIDAÇÃO e TESTE
X_val_tfidf = tfidf_vect.transform(val_clean['message_preprocessed'])
X_test_tfidf = tfidf_vect.transform(test_clean['message_preprocessed'])

# Rótulos (y)
y_train = train_clean['label']
y_val = val_clean['label']
y_test = test_clean['label']

print("=== VETORIZAÇÃO TF-IDF CONCLUÍDA ===")
print(f"Shape Treino:     {X_train_tfidf.shape}")
print(f"Shape Validação:  {X_val_tfidf.shape}")
print(f"Shape Teste Real: {X_test_tfidf.shape}")

=== VETORIZAÇÃO TF-IDF CONCLUÍDA ===
Shape Treino:     (39995, 10000)
Shape Validação:  (4986, 10000)
Shape Teste Real: (4986, 10000)


## Word2Vec

In [16]:
from gensim.models import Word2Vec
import numpy as np

# 1. Preparar os tokens para o Word2Vec (apenas treino)
train_sentences = [text.split() for text in train_clean['message_preprocessed']]

# 2. Treinar o modelo Word2Vec
# vector_size=100, window=5, min_count=2 para cobrir gírias recorrentes
w2v_model = Word2Vec(sentences=train_sentences, vector_size=100, window=5, min_count=2, workers=4, seed=42)

# 3. Função para transformar mensagem em vetor médio (Sentença -> Vetor)
def get_average_word2vec(tokens, model, vector_size):
    valid_vectors = [model.wv[word] for word in tokens if word in model.wv]
    if not valid_vectors:
        return np.zeros(vector_size)
    return np.mean(valid_vectors, axis=0)

# 4. Vetorizar os datasets (Treino, Validação e Teste)
X_train_w2v = np.array([get_average_word2vec(text.split(), w2v_model, 100) for text in train_clean['message_preprocessed']])
X_val_w2v = np.array([get_average_word2vec(text.split(), w2v_model, 100) for text in val_clean['message_preprocessed']])
X_test_w2v = np.array([get_average_word2vec(text.split(), w2v_model, 100) for text in test_clean['message_preprocessed']])

print("=== VETORIZAÇÃO WORD2VEC CONCLUÍDA ===")
print(f"Shape Treino W2V: {X_train_w2v.shape}")
print(f"Shape Teste W2V:  {X_test_w2v.shape}")

=== VETORIZAÇÃO WORD2VEC CONCLUÍDA ===
Shape Treino W2V: (39995, 100)
Shape Teste W2V:  (4986, 100)


In [15]:
from gensim.models import Word2Vec
import numpy as np

# 1. Preparar os tokens para o Word2Vec
train_sentences = [text.split() for text in train_clean['message_preprocessed']]

# 2. Treinar o modelo Word2Vec
# vector_size=100 (dimensões), window=5, min_count=2 para ignorar palavras muito raras
w2v_model = Word2Vec(sentences=train_sentences, vector_size=100, window=5, min_count=2, workers=4, seed=42)

# 3. Função para transformar mensagem em vetor médio
def get_average_word2vec(tokens, model, vector_size):
    valid_vectors = [model.wv[word] for word in tokens if word in model.wv]
    if not valid_vectors:
        return np.zeros(vector_size)
    return np.mean(valid_vectors, axis=0)

# 4. Vetorizar os datasets
X_train_w2v = np.array([get_average_word2vec(text.split(), w2v_model, 100) for text in train_clean['message_preprocessed']])
X_val_w2v = np.array([get_average_word2vec(text.split(), w2v_model, 100) for text in val_clean['message_preprocessed']])
X_test_w2v = np.array([get_average_word2vec(text.split(), w2v_model, 100) for text in test_clean['message_preprocessed']])

print("=== VETORIZAÇÃO WORD2VEC CONCLUÍDA ===")
print(f"Shape Treino W2V: {X_train_w2v.shape}")
print(f"Exemplo de vetor (primeiros 5 elementos): {X_train_w2v[0][:5]}")

=== VETORIZAÇÃO WORD2VEC CONCLUÍDA ===
Shape Treino W2V: (39995, 100)
Exemplo de vetor (primeiros 5 elementos): [ 0.13691926 -0.07566585 -0.02604797  0.16971526 -0.36329347]


# Modelos de Classificação

## TF-IDF

In [10]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, confusion_matrix, f1_score

# 1. Regressão Logística
model_lr = LogisticRegression(max_iter=1000, class_weight='balanced', random_state=42)
model_lr.fit(X_train_tfidf, y_train)

# Predição e Avaliação
y_pred_lr = model_lr.predict(X_test_tfidf)

print("=== BASELINE: REGRESSÃO LOGÍSTICA ===")
print(classification_report(y_test, y_pred_lr, target_names=list(class_mapping.values())))
print(f"Macro F1-Score: {f1_score(y_test, y_pred_lr, average='macro'):.4f}")

=== BASELINE: REGRESSÃO LOGÍSTICA ===
                       precision    recall  f1-score   support

            Non-Toxic       0.95      0.85      0.90      4044
  Insults and Flaming       0.67      0.68      0.67       681
Other Offensive Texts       0.26      0.61      0.36       222
  Hate and Harassment       0.11      0.38      0.17        29
              Threats       0.08      0.38      0.13         8
            Extremism       0.25      0.50      0.33         2

             accuracy                           0.82      4986
            macro avg       0.39      0.57      0.43      4986
         weighted avg       0.88      0.82      0.84      4986

Macro F1-Score: 0.4273


In [11]:
from sklearn.svm import LinearSVC

# 2. SVM (Linear)
model_svm = LinearSVC(class_weight='balanced', random_state=42, max_iter=2000)
model_svm.fit(X_train_tfidf, y_train)

# Predição e Avaliação
y_pred_svm = model_svm.predict(X_test_tfidf)

print("=== BASELINE: SVM (LINEAR) ===")
print(classification_report(y_test, y_pred_svm, target_names=list(class_mapping.values())))
print(f"Macro F1-Score: {f1_score(y_test, y_pred_svm, average='macro'):.4f}")

=== BASELINE: SVM (LINEAR) ===
                       precision    recall  f1-score   support

            Non-Toxic       0.94      0.93      0.93      4044
  Insults and Flaming       0.75      0.67      0.71       681
Other Offensive Texts       0.35      0.47      0.40       222
  Hate and Harassment       0.24      0.41      0.30        29
              Threats       0.11      0.12      0.12         8
            Extremism       0.00      0.00      0.00         2

             accuracy                           0.87      4986
            macro avg       0.40      0.44      0.41      4986
         weighted avg       0.88      0.87      0.87      4986

Macro F1-Score: 0.4112


/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


In [12]:
from sklearn.naive_bayes import MultinomialNB

# 3. Naive Bayes (Multinomial)
model_nb = MultinomialNB()
model_nb.fit(X_train_tfidf, y_train)

# Predição e Avaliação
y_pred_nb = model_nb.predict(X_test_tfidf)

print("=== BASELINE: NAIVE BAYES ===")
print(classification_report(y_test, y_pred_nb, target_names=list(class_mapping.values())))
print(f"Macro F1-Score: {f1_score(y_test, y_pred_nb, average='macro'):.4f}")

=== BASELINE: NAIVE BAYES ===
                       precision    recall  f1-score   support

            Non-Toxic       0.88      0.99      0.93      4044
  Insults and Flaming       0.83      0.52      0.64       681
Other Offensive Texts       0.84      0.07      0.13       222
  Hate and Harassment       0.00      0.00      0.00        29
              Threats       0.00      0.00      0.00         8
            Extremism       0.00      0.00      0.00         2

             accuracy                           0.88      4986
            macro avg       0.43      0.26      0.28      4986
         weighted avg       0.87      0.88      0.85      4986

Macro F1-Score: 0.2843


/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


In [13]:
from sklearn.ensemble import RandomForestClassifier

# 4. Random Forest
model_rf = RandomForestClassifier(n_estimators=100, class_weight='balanced', random_state=42, n_jobs=-1)
model_rf.fit(X_train_tfidf, y_train)

# Predição e Avaliação
y_pred_rf = model_rf.predict(X_test_tfidf)

print("=== BASELINE: RANDOM FOREST ===")
print(classification_report(y_test, y_pred_rf, target_names=list(class_mapping.values())))
print(f"Macro F1-Score: {f1_score(y_test, y_pred_rf, average='macro'):.4f}")

=== BASELINE: RANDOM FOREST ===
                       precision    recall  f1-score   support

            Non-Toxic       0.92      0.86      0.89      4044
  Insults and Flaming       0.76      0.63      0.69       681
Other Offensive Texts       0.48      0.35      0.40       222
  Hate and Harassment       0.29      0.31      0.30        29
              Threats       0.05      0.12      0.07         8
            Extremism       0.00      0.00      0.00         2

             accuracy                           0.80      4986
            macro avg       0.42      0.38      0.39      4986
         weighted avg       0.87      0.80      0.83      4986

Macro F1-Score: 0.3919


## Word2Vec

In [25]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, f1_score

# 1. Regressão Logística (Word2Vec)
model_lr_w2v = LogisticRegression(max_iter=1000, class_weight='balanced', random_state=42)
model_lr_w2v.fit(X_train_w2v, y_train)

y_pred_lr_w2v = model_lr_w2v.predict(X_test_w2v)

print("=== BASELINE W2V: REGRESSÃO LOGÍSTICA ===")
print(classification_report(y_test, y_pred_lr_w2v, target_names=list(class_mapping.values())))
print(f"Macro F1-Score: {f1_score(y_test, y_pred_lr_w2v, average='macro'):.4f}")

=== BASELINE W2V: REGRESSÃO LOGÍSTICA ===
                       precision    recall  f1-score   support

            Non-Toxic       0.94      0.42      0.58      4044
  Insults and Flaming       0.53      0.47      0.50       681
Other Offensive Texts       0.13      0.48      0.21       222
  Hate and Harassment       0.03      0.41      0.05        29
              Threats       0.01      0.62      0.02         8
            Extremism       0.00      0.00      0.00         2

             accuracy                           0.43      4986
            macro avg       0.27      0.40      0.23      4986
         weighted avg       0.84      0.43      0.55      4986

Macro F1-Score: 0.2253


In [26]:
from sklearn.svm import LinearSVC

# 2. SVM (Word2Vec)
model_svm_w2v = LinearSVC(class_weight='balanced', random_state=42, max_iter=5000)
model_svm_w2v.fit(X_train_w2v, y_train)

y_pred_svm_w2v = model_svm_w2v.predict(X_test_w2v)

print("=== BASELINE W2V: SVM (LINEAR) ===")
print(classification_report(y_test, y_pred_svm_w2v, target_names=list(class_mapping.values())))
print(f"Macro F1-Score: {f1_score(y_test, y_pred_svm_w2v, average='macro'):.4f}")

=== BASELINE W2V: SVM (LINEAR) ===
                       precision    recall  f1-score   support

            Non-Toxic       0.88      0.93      0.90      4044
  Insults and Flaming       0.67      0.43      0.52       681
Other Offensive Texts       0.41      0.28      0.33       222
  Hate and Harassment       0.00      0.00      0.00        29
              Threats       0.04      0.12      0.06         8
            Extremism       0.00      0.00      0.00         2

             accuracy                           0.82      4986
            macro avg       0.33      0.29      0.30      4986
         weighted avg       0.83      0.82      0.82      4986

Macro F1-Score: 0.3034


In [27]:
from sklearn.naive_bayes import GaussianNB

# 3. Naive Bayes (Gaussian para vetores densos)
model_nb_w2v = GaussianNB()
model_nb_w2v.fit(X_train_w2v, y_train)

y_pred_nb_w2v = model_nb_w2v.predict(X_test_w2v)

print("=== BASELINE W2V: NAIVE BAYES (GAUSSIAN) ===")
print(classification_report(y_test, y_pred_nb_w2v, target_names=list(class_mapping.values())))
print(f"Macro F1-Score: {f1_score(y_test, y_pred_nb_w2v, average='macro'):.4f}")

=== BASELINE W2V: NAIVE BAYES (GAUSSIAN) ===
                       precision    recall  f1-score   support

            Non-Toxic       0.93      0.10      0.17      4044
  Insults and Flaming       0.18      0.41      0.25       681
Other Offensive Texts       0.08      0.11      0.09       222
  Hate and Harassment       0.03      0.45      0.05        29
              Threats       0.00      0.25      0.00         8
            Extremism       0.00      0.00      0.00         2

             accuracy                           0.14      4986
            macro avg       0.20      0.22      0.09      4986
         weighted avg       0.78      0.14      0.18      4986

Macro F1-Score: 0.0947


In [28]:
from sklearn.ensemble import RandomForestClassifier

# 4. Random Forest (Word2Vec)
model_rf_w2v = RandomForestClassifier(n_estimators=100, class_weight='balanced', random_state=42, n_jobs=-1)
model_rf_w2v.fit(X_train_w2v, y_train)

y_pred_rf_w2v = model_rf_w2v.predict(X_test_w2v)

print("=== BASELINE W2V: RANDOM FOREST ===")
print(classification_report(y_test, y_pred_rf_w2v, target_names=list(class_mapping.values())))
print(f"Macro F1-Score: {f1_score(y_test, y_pred_rf_w2v, average='macro'):.4f}")

=== BASELINE W2V: RANDOM FOREST ===
                       precision    recall  f1-score   support

            Non-Toxic       0.88      0.90      0.89      4044
  Insults and Flaming       0.77      0.41      0.53       681
Other Offensive Texts       0.51      0.34      0.41       222
  Hate and Harassment       0.38      0.17      0.24        29
              Threats       0.06      0.12      0.08         8
            Extremism       0.00      0.00      0.00         2

             accuracy                           0.80      4986
            macro avg       0.43      0.32      0.36      4986
         weighted avg       0.84      0.80      0.81      4986

Macro F1-Score: 0.3573
